<a href="https://colab.research.google.com/github/Slavena1/Softuni-Deep-Learning-Final-Project/blob/main/notebook/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Child, Learner, Machine: Benchmarking Real Language Models Against Human Mandarin Acquisition
##### *Author: Slavena Peneva-Kargiou*
##### *Course: SoftUni Deep Learning*
##### *Instructor: Yordan Darakchiev*
##### *Date: August 2026*

Builds on: [SoftUni Machine Learning Final Project](https://github.com/Slavena1/Softuni-Machine-Learning-Final-Project)

In [11]:
%cd /content/Softuni-Deep-Learning-Final-Project/notebook

/content/Softuni-Deep-Learning-Final-Project/notebook


In [12]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, mannwhitneyu

import torch

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score, classification_report
)

# LLM API clients — add once Section 4.5 is implemented (after the Aug 3 exercise)
# import anthropic
# import openai

## 1. Introduction

My Machine Learning project asked a simple question: when Mandarin speakers acquire new vocabulary - as children learning their first language, or as adults learning a second one, what makes a word easy or hard? The answer turned out to depend on who's doing the learning. Children lean on concreteness: words for things you can point at and picture come first. Adult learners lean on frequency: the words you hear most often stick first. A third group sat in that project too, but simply as an assumption rather than a measurement - "the language model" standing in as a kind of frequency-maximizing learner, represented just by a static word-frequency table pulled from the Chinese BabyLM corpus.

This project asks what happens if I stop assuming and start testing.

**The core idea:** take real language models, not just a frequency table, and ask them directly, the way you'd quiz a learner, which Mandarin words they seem to know well and which they stumble on. Then compare that pattern against real human data: which words Mandarin-speaking children learn first, and which words are ranked as harder on the HSK proficiency scale that adult learners study against. Do today's language models look more like the concreteness driven child, the frequency driven adult learner, or something else entirely? And does that answer change depending on *how* you ask - a small model's internal probabilities, or a large commercial model's actual behavior when prompted?

**Motivation:** while looking into the Chinese BabyLM Challenge's own evaluation pipeline - the resource my ML project drew its corpus from, I found that its three evaluation tracks (grammar, character structure, brain-signal alignment) do not include anything like an acquisition-order comparison, even though the general English language BabyLM pipeline has exactly this kind of task. This is something that I feel in a good position to try exploring, given that the ML project already built half the ingredients. I reached out to the team behind that pipeline in July 2026 to ask whether this seemed like a real gap worth exploring before committing further time to it (*no answer yet, fill in accordingly*).

**What is new here, versus the ML project:** that project used Ridge and Random Forest regression on three hand-picked features (frequency, concreteness, word length) to find the pattern in the first place. This project three additional things:
- replaces the frequency-table proxy for "the language model" with direct behavioral testing of real LLMs, via API;
- adds an actual neural architecture comparison - a dense network trained on pretrained transformer embeddings, using the same frozen-backbone transfer-learning pattern this course teaches for vision and language models, evaluated against the earlier Ridge/RF baseline rather than replacing it outright;
- treats reliability engineering (caching, retries, cost tracking) as part of the methodology, not as an afterthought.

The rest of this notebook is organized to make that comparison as fair and transparent as I can manage: the same data, the same train/ test discipline, and an explicit accounting of which results I am confident in and which are still open questions.



## 2. Related Work
The BabyLM Chalenge (Warstadt et al., 2023) anchors the practical part of this project. It asks what happens if a language model is trained on roughly the amount of text a child actually hears growing up. The English evaluation pipeline built for it includes an age-of-acquisition (AoA) prediction task: check whether a trained model's behavior on a word correlates with the age at which real children acquire it.

The 2026 BabyLM Workshop introduced a new multilingual track built on BabyBabelLM, a training data resource explicitly covering English, Dutch and Chinese (Jumelet et al., 2026), with the year's theme being "going beyond English" - a direct invitation for this kind of non-English acquisition work.  

The Chinese BabyLM Challenge, colocated with NLPCC 2026, extends this to Mandarin and it is where my ML project's corpus came from. Its own evaluation pipeline takes a different shape: grammatical acceptability (NLU), character-level structural knowledge (Hanzi), and alignment with human brain recordings (Cog), but nothing resembling the English pipeline's AoA-prediction task. That's the gap this project tries to fill - I checked this reading of their pipeline directly against their published evaluation code before committing to the idea.

## 3. Data

Same data as the ML project, reloaded here via the same loader functions (src/data_prep.py, unchanged).

In [16]:
import sys
sys.path.insert(0, '/content/Softuni-Deep-Learning-Final-Project/src')

import data_prep as dp
import features as ft

DATA_DIR = '/content/drive/MyDrive/SoftUni-Machine-Learning-Final-Project-Datasets'

df_aoa = dp.load_wordbank(f'{DATA_DIR}/wordbank_mandarin_items.csv')
hsk_paths = {level: f'{DATA_DIR}/hsk{level}.csv' for level in range(1, 7)}
df_hsk = dp.load_hsk(hsk_paths)
df_freq = dp.load_babylm_frequencies(
    f'{DATA_DIR}/babylm_zh_frequencies.csv',
    parquet_path=f'{DATA_DIR}/train-00000-of-00001.parquet'
)
df_xuli = dp.load_xuli_concreteness(
    f'{DATA_DIR}/Concretenss_Ratings_of_9877_Two_Character_Chinese_Words.xlsx')
df_liu = dp.load_liu_concreteness(f'{DATA_DIR}/liu_2007_single_char.txt')

print(f"Child AoA words: {len(df_aoa)}")
print(f"HSK words: {len(df_hsk)}")
print(f"Frequency table: {len(df_freq)} words")
print(f"Concreteness norms: {len(df_xuli)} (Xu & Li) + {len(df_liu)} (Liu et al.)")

Loaded pre-computed frequencies: 606,029 unique words
Child AoA words: 799
HSK words: 4993
Frequency table: 606029 words
Concreteness norms: 9877 (Xu & Li) + 2356 (Liu et al.)


In [17]:
import os
print(os.listdir(DATA_DIR))

['wordbank_mandarin_items.csv', 'hsk1.csv', 'hsk2.csv', 'hsk3.csv', 'hsk4.csv', 'hsk5.csv', 'hsk6.csv', 'train-00000-of-00001.parquet', 'Concretenss_Ratings_of_9877_Two_Character_Chinese_Words.xlsx', 'liu_2007_single_char.txt', 'babylm_zh_frequencies.csv']


In [18]:
# Concreteness merging: Xu & Li (two-character) + Liu et al. (single-character)
df_xuli['concreteness_norm'] = dp.normalize_concreteness(df_xuli['concreteness'])
df_liu['concreteness_norm'] = dp.normalize_concreteness(df_liu['concreteness_raw'])

df_concrete_combined = pd.concat([
    df_xuli[['word', 'concreteness_norm']],
    df_liu[['word', 'concreteness_norm']]
], ignore_index=True)

# Reduplicated words (e.g. 妈妈, 爸爸): concreteness derived from the base character,
# since these carry no independent concreteness rating in either source dataset
liu_lookup = dict(zip(df_liu['word'], df_liu['concreteness_norm']))
missing_conc_words = df_aoa[~df_aoa['word'].isin(df_concrete_combined['word'])]['word'].tolist()
reduplications = [w for w in missing_conc_words if dp.is_reduplication(w)]

reduplication_rows = []
for word in reduplications:
    base_char = word[0]
    if base_char in liu_lookup:
        reduplication_rows.append({'word': word, 'concreteness_norm': liu_lookup[base_char]})
df_reduplications = pd.DataFrame(reduplication_rows)

df_concrete_final = pd.concat(
    [df_concrete_combined, df_reduplications], ignore_index=True
).drop_duplicates(subset='word', keep='first')

print(f"Final concreteness dataset: {len(df_concrete_final):,} unique words")

Final concreteness dataset: 12,250 unique words


In [19]:
df_child = dp.build_dataset(df_aoa.dropna(subset=['aoa']), 'aoa', df_freq, df_concrete_final)
df_adult = dp.build_dataset(df_hsk, 'hsk_level', df_freq, df_concrete_final)

FEATURES = ['log_frequency', 'concreteness', 'word_length']

df_child_analysis = df_child.dropna(subset=['aoa'] + FEATURES).copy().reset_index(drop=True)
df_adult_analysis = df_adult.dropna(subset=['hsk_level'] + FEATURES).copy().reset_index(drop=True)

print(f"Child analysis dataset: {len(df_child_analysis)} words")
print(f"Adult analysis dataset: {len(df_adult_analysis)} words")

Child analysis dataset: 504 words
Adult analysis dataset: 3269 words


## 4. Methods

### 4.1 Baseline: Ridge/ Random Forest (from the ML project)

In [22]:
import baseline_models as bm

# Split on raw (unscaled) features - standardization happens inside each
# model's pipeline, fit on training data only
X_c_train, X_c_test, y_c_train, y_c_test, idx_c_train, idx_c_test = bm.split_raw(
    df_child_analysis, FEATURES, 'aoa')

X_a_train, X_a_test, y_a_train, y_a_test, idx_a_train, idx_a_test = bm.split_raw(
    df_adult_analysis, FEATURES, 'hsk_level')

print(f"Child train/test sizes: {len(X_c_train)} / {len(X_c_test)}")
print(f"Adult train/test sizes: {len(X_a_train)} / {len(X_a_test)}")

Child train/test sizes: 378 / 126
Adult train/test sizes: 2451 / 818


In [23]:
param_grid = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}
cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid_child = bm.fit_ridge(X_c_train, y_c_train, param_grid, cv=cv)
ridge_child = grid_child.best_estimator_
y_c_pred = ridge_child.predict(X_c_test)
cv_scores_child = cross_val_score(ridge_child, X_c_train, y_c_train, cv=cv, scoring='r2')

print(f"Child AoA - Ridge (alpha={grid_child.best_params_['ridge__alpha']}):")
print(f"  R2: {r2_score(y_c_test, y_c_pred):.3f}  |  CV R2: {cv_scores_child.mean():.3f} +/- {cv_scores_child.std():.3f}")

grid_adult = bm.fit_ridge(X_a_train, y_a_train, param_grid, cv=cv)
ridge_adult = grid_adult.best_estimator_
y_a_pred = ridge_adult.predict(X_a_test)
cv_scores_adult = cross_val_score(ridge_adult, X_a_train, y_a_train, cv=cv, scoring='r2')

print(f"\nAdult HSK - Ridge (alpha={grid_adult.best_params_['ridge__alpha']}):")
print(f"  R2: {r2_score(y_a_test, y_a_pred):.3f}  |  CV R2: {cv_scores_adult.mean():.3f} +/- {cv_scores_adult.std():.3f}")

Child AoA - Ridge (alpha=10):
  R2: -0.155  |  CV R2: 0.312 +/- 0.106

Adult HSK - Ridge (alpha=10):
  R2: 0.509  |  CV R2: 0.531 +/- 0.028


Random Forest is run as a **classifier**, not a regressor, on a binned version of each target — early/middle/late (via quantile split) for child AoA, and the six HSK levels directly for adult. This gives a discrete, model-native feature - importance view to compare against Ridge's coefficients in Section 5, alongside the continuous R² reported above.

In [24]:
df_child_analysis['aoa_class'] = pd.qcut(df_child_analysis['aoa'], q=3, labels=['early', 'middle', 'late'])
X_c_tr_c, X_c_te_c, y_c_tr_c, y_c_te_c, idx_c_tr_c, idx_c_te_c = bm.split_raw(
    df_child_analysis, FEATURES, 'aoa_class', stratify_col='aoa_class')

rf_child = bm.fit_rf_classifier(X_c_tr_c, y_c_tr_c)
print("Child AoA - Random Forest (early/middle/late):")
print(classification_report(y_c_te_c, rf_child.predict(X_c_te_c)))

df_adult_analysis['hsk_class'] = df_adult_analysis['hsk_level'].astype(int)
X_a_tr_c, X_a_te_c, y_a_tr_c, y_a_te_c, idx_a_tr_c, idx_a_te_c = bm.split_raw(
    df_adult_analysis, FEATURES, 'hsk_class', stratify_col='hsk_class')

rf_adult = bm.fit_rf_classifier(X_a_tr_c, y_a_tr_c)
print("\nHSK Level - Random Forest (6 classes):")
print(classification_report(y_a_te_c, rf_adult.predict(X_a_te_c),
                             target_names=[f'HSK {i}' for i in range(1, 7)]))

Child AoA - Random Forest (early/middle/late):
              precision    recall  f1-score   support

       early       0.61      0.59      0.60        66
        late       0.26      0.24      0.25        21
      middle       0.33      0.36      0.34        39

    accuracy                           0.46       126
   macro avg       0.40      0.40      0.40       126
weighted avg       0.46      0.46      0.46       126


HSK Level - Random Forest (6 classes):
              precision    recall  f1-score   support

       HSK 1       0.25      0.24      0.24        21
       HSK 2       0.24      0.19      0.21        27
       HSK 3       0.29      0.28      0.28        58
       HSK 4       0.32      0.28      0.30       112
       HSK 5       0.41      0.39      0.40       228
       HSK 6       0.71      0.78      0.74       372

    accuracy                           0.53       818
   macro avg       0.37      0.36      0.36       818
weighted avg       0.52      0.53      0.52 

In [25]:
param_grid_lasso = {'alpha': [0.001, 0.005, 0.01, 0.05, 0.1, 0.5]}

lasso_child = bm.fit_lasso(X_c_train, y_c_train, param_grid_lasso, cv=cv)
lasso_adult = bm.fit_lasso(X_a_train, y_a_train, param_grid_lasso, cv=cv)

print(f"Lasso - Child (alpha={lasso_child.best_params_['lasso__alpha']}):")
for f_name, coef in zip(FEATURES, lasso_child.best_estimator_.named_steps['lasso'].coef_):
    print(f"  {f_name:<15}: {coef:>8.4f}  [{'RETAINED' if coef != 0 else 'ZEROED OUT'}]")

print(f"\nLasso - Adult (alpha={lasso_adult.best_params_['lasso__alpha']}):")
for f_name, coef in zip(FEATURES, lasso_adult.best_estimator_.named_steps['lasso'].coef_):
    print(f"  {f_name:<15}: {coef:>8.4f}  [{'RETAINED' if coef != 0 else 'ZEROED OUT'}]")

Lasso - Child (alpha=0.01):
  log_frequency  :  -1.0094  [RETAINED]
  concreteness   :  -1.0399  [RETAINED]
  word_length    :   0.9820  [RETAINED]

Lasso - Adult (alpha=0.001):
  log_frequency  :  -0.8302  [RETAINED]
  concreteness   :  -0.2270  [RETAINED]
  word_length    :   0.0883  [RETAINED]


### 4.2 Dense NN on hand-picked features

In [26]:
from sklearn.preprocessing import StandardScaler
import neural_models as nm

X_c_tr2, X_c_val, y_c_tr2, y_c_val = nm.make_train_val_split(X_c_train, y_c_train)
X_a_tr2, X_a_val, y_a_tr2, y_a_val = nm.make_train_val_split(X_a_train, y_a_train)

scaler_c = StandardScaler().fit(X_c_tr2)
scaler_a = StandardScaler().fit(X_a_tr2)

nn_child, hist_child = nm.train_dense_nn(
    scaler_c.transform(X_c_tr2), y_c_tr2, scaler_c.transform(X_c_val), y_c_val)
metrics_nn_child = nm.evaluate_nn(nn_child, scaler_c.transform(X_c_test), y_c_test)

nn_adult, hist_adult = nm.train_dense_nn(
    scaler_a.transform(X_a_tr2), y_a_tr2, scaler_a.transform(X_a_val), y_a_val)
metrics_nn_adult = nm.evaluate_nn(nn_adult, scaler_a.transform(X_a_test), y_a_test)

print(f"Child AoA  - Dense NN (hand-picked features): R2 = {metrics_nn_child['r2']:.3f}")
print(f"Adult HSK  - Dense NN (hand-picked features): R2 = {metrics_nn_adult['r2']:.3f}")

Early stopping at epoch 189 (best val loss: 5.2299)
Early stopping at epoch 89 (best val loss: 0.6467)
Child AoA  - Dense NN (hand-picked features): R2 = -0.302
Adult HSK  - Dense NN (hand-picked features): R2 = 0.546


In [27]:
import embeddings as emb

model, tokenizer = emb.load_pretrained_model("hfl/chinese-macbert-base")
print("MacBERT loaded and frozen.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/269k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

...

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  412MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: hfl/chinese-macbert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MacBERT loaded and frozen.


In [29]:
# Build embeddings in the same order as the analysis dataframes
X_c_embeddings = emb.build_embedding_matrix(
    df_child_analysis['word'].tolist(),
    model, tokenizer, pooling="mean"
)

X_a_embeddings = emb.build_embedding_matrix(
    df_adult_analysis['word'].tolist(),
    model, tokenizer, pooling="mean"
)

print("Child embeddings shape:", X_c_embeddings.shape)
print("Adult embeddings shape:", X_a_embeddings.shape)

  0/504 words embedded...
  100/504 words embedded...
  200/504 words embedded...
  300/504 words embedded...
  400/504 words embedded...
  500/504 words embedded...
  504/504 done.
  0/3269 words embedded...
  100/3269 words embedded...
  200/3269 words embedded...
  300/3269 words embedded...
  400/3269 words embedded...
  500/3269 words embedded...
  600/3269 words embedded...
  700/3269 words embedded...
  800/3269 words embedded...
  900/3269 words embedded...
  1000/3269 words embedded...
  1100/3269 words embedded...
  1200/3269 words embedded...
  1300/3269 words embedded...
  1400/3269 words embedded...
  1500/3269 words embedded...
  1600/3269 words embedded...
  1700/3269 words embedded...
  1800/3269 words embedded...
  1900/3269 words embedded...
  2000/3269 words embedded...
  2100/3269 words embedded...
  2200/3269 words embedded...
  2300/3269 words embedded...
  2400/3269 words embedded...
  2500/3269 words embedded...
  2600/3269 words embedded...
  2700/3269 words em

### 4.3 Transfer learning: pretrained transformer embeddings

In [30]:
X_c_emb_train = X_c_embeddings[idx_c_train]
X_c_emb_test = X_c_embeddings[idx_c_test]
X_a_emb_train = X_a_embeddings[idx_a_train]
X_a_emb_test = X_a_embeddings[idx_a_test]

X_c_emb_tr2, X_c_emb_val, y_c_tr2b, y_c_valb = nm.make_train_val_split(X_c_emb_train, y_c_train)
X_a_emb_tr2, X_a_emb_val, y_a_tr2b, y_a_valb = nm.make_train_val_split(X_a_emb_train, y_a_train)

scaler_c_emb = StandardScaler().fit(X_c_emb_tr2)
scaler_a_emb = StandardScaler().fit(X_a_emb_tr2)

nn_child_emb, hist_child_emb = nm.train_dense_nn(
    scaler_c_emb.transform(X_c_emb_tr2), y_c_tr2b,
    scaler_c_emb.transform(X_c_emb_val), y_c_valb, dropout=0.4)
metrics_nn_child_emb = nm.evaluate_nn(nn_child_emb, scaler_c_emb.transform(X_c_emb_test), y_c_test)

nn_adult_emb, hist_adult_emb = nm.train_dense_nn(
    scaler_a_emb.transform(X_a_emb_tr2), y_a_tr2b,
    scaler_a_emb.transform(X_a_emb_val), y_a_valb, dropout=0.4)
metrics_nn_adult_emb = nm.evaluate_nn(nn_adult_emb, scaler_a_emb.transform(X_a_emb_test), y_a_test)

print(f"Child AoA  - Dense NN (MacBERT embeddings): R2 = {metrics_nn_child_emb['r2']:.3f}")
print(f"Adult HSK  - Dense NN (MacBERT embeddings): R2 = {metrics_nn_adult_emb['r2']:.3f}")

Early stopping at epoch 49 (best val loss: 10.7810)
Early stopping at epoch 112 (best val loss: 0.8577)
Child AoA  - Dense NN (MacBERT embeddings): R2 = -0.494
Adult HSK  - Dense NN (MacBERT embeddings): R2 = 0.470


### 4.4 Attention weight analysis

???

### 4.5 LLM API testing: sentence completion + minimal pairs

### 4.6 Cost tracking

## 5. Results

## 6. Uncertainty Analysis

## 7. Error Analysis

## 8. Discussion

## 9. Limitations & Future Work

## 10. Conclusion

## 11. References

- Warstadt, A., et al. (2023). Findings of the BabyLM Challenge: Sample-efficient pretraining on developmentally plausible corpora. *Proceedings of the BabyLM Challenge*.
- Jumelet, J., et al. (2026). BabyBabelLM: A multilingual benchmark of developmentally plausible training data. *EACL 2026*. (verify full author list/formatting before final submission)
- Chinese BabyLM Challenge (2026). Co-located with NLPCC 2026. chinese-babylm.github.io


test commit